In [3]:
from astropy.io import fits
import numpy as np
from scipy.ndimage import gaussian_filter, convolve1d

input_fits = "/Users/golta/Neb_det/data/M33_MAPS.fits"
output_fits = "M33_4channel_NGC253_distance.fits"

line_extensions = [
    "HB4861_FLUX",
    "OIII5006_FLUX",
    "HA6562_FLUX",
    "NII6583_FLUX",
]
error_extensions = [f"{name}_ERR" for name in line_extensions]

# Distances in Mpc
distance_m33 = 0.84
distance_ngc253 = 3.50

# Angular-size scaling and expected integrated-flux scaling
angular_scale = distance_m33 / distance_ngc253
expected_flux_scale = angular_scale**2

# Adjust these to match your data.
pixel_scale_arcsec = 0.20
input_fwhm_arcsec = 0.80
target_fwhm_arcsec = 0.80


def rescale_flux_and_error_about_center(flux, error, scale, center_yx=None):
    """Bilinearly shrink a flux map and propagate independent variances.

    For interpolation coefficients a_i, the resampled flux is sum(a_i f_i)
    and its variance is sum(a_i**2 sigma_i**2). NaN neighbours are omitted
    and the remaining coefficients are renormalized.
    """
    flux = np.asarray(flux, dtype=np.float32)
    error = np.asarray(error, dtype=np.float32)
    if flux.shape != error.shape or flux.ndim != 2:
        raise ValueError(f"Flux/error shapes must be matching 2-D arrays, got {flux.shape} and {error.shape}")

    ny, nx = flux.shape
    center = (
        np.array([(ny - 1) / 2.0, (nx - 1) / 2.0], dtype=np.float64)
        if center_yx is None else np.asarray(center_yx, dtype=np.float64)
    )
    yy, xx = np.indices((ny, nx), dtype=np.float32)
    yin = center[0] + (yy - center[0]) / scale
    xin = center[1] + (xx - center[1]) / scale
    inside = (yin >= 0) & (yin <= ny - 1) & (xin >= 0) & (xin <= nx - 1)

    y0 = np.clip(np.floor(yin).astype(np.int32), 0, ny - 1)
    x0 = np.clip(np.floor(xin).astype(np.int32), 0, nx - 1)
    y1 = np.minimum(y0 + 1, ny - 1)
    x1 = np.minimum(x0 + 1, nx - 1)
    dy = yin - y0
    dx = xin - x0

    flux_num = np.zeros((ny, nx), dtype=np.float64)
    variance_num = np.zeros((ny, nx), dtype=np.float64)
    weight_sum = np.zeros((ny, nx), dtype=np.float64)
    neighbours = [
        (y0, x0, (1 - dy) * (1 - dx)),
        (y0, x1, (1 - dy) * dx),
        (y1, x0, dy * (1 - dx)),
        (y1, x1, dy * dx),
    ]
    for yi, xi, coefficient in neighbours:
        f = flux[yi, xi]
        e = error[yi, xi]
        good = inside & np.isfinite(f) & np.isfinite(e) & (e >= 0)
        a = np.where(good, coefficient, 0.0)
        flux_num += a * np.where(good, f, 0.0)
        variance_num += a**2 * np.where(good, e**2, 0.0)
        weight_sum += a

    valid = weight_sum > 0.01
    scaled_flux = np.full((ny, nx), np.nan, dtype=np.float32)
    scaled_error = np.full((ny, nx), np.nan, dtype=np.float32)
    scaled_flux[valid] = (flux_num[valid] / weight_sum[valid]).astype(np.float32)
    scaled_error[valid] = (np.sqrt(variance_num[valid]) / weight_sum[valid]).astype(np.float32)
    return scaled_flux, scaled_error


def gaussian_kernel_1d(sigma_pixels, truncate=4.0):
    radius = int(truncate * sigma_pixels + 0.5)
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    kernel = np.exp(-0.5 * (x / sigma_pixels)**2)
    return kernel / kernel.sum()


def convolve_flux_and_error_preserving_nans(flux, error, sigma_pixels):
    """Apply a Gaussian PSF and propagate per-pixel independent variance."""
    if sigma_pixels <= 0:
        return flux, error

    finite = np.isfinite(flux) & np.isfinite(error) & (error >= 0)
    mask = finite.astype(np.float32)
    denominator = gaussian_filter(mask, sigma_pixels, mode="constant", cval=0.0)
    flux_num = gaussian_filter(
        np.where(finite, flux, 0.0), sigma_pixels, mode="constant", cval=0.0
    )

    # If y_j = sum_i(k_i x_i), Var(y_j) = sum_i(k_i**2 Var(x_i)).
    kernel_sq = gaussian_kernel_1d(sigma_pixels)**2
    variance = np.where(finite, error**2, 0.0).astype(np.float64)
    variance_num = convolve1d(variance, kernel_sq, axis=0, mode="constant", cval=0.0)
    variance_num = convolve1d(variance_num, kernel_sq, axis=1, mode="constant", cval=0.0)

    valid = denominator > 1e-3
    result_flux = np.full(flux.shape, np.nan, dtype=np.float32)
    result_error = np.full(flux.shape, np.nan, dtype=np.float32)
    result_flux[valid] = (flux_num[valid] / denominator[valid]).astype(np.float32)
    result_error[valid] = (np.sqrt(variance_num[valid]) / denominator[valid]).astype(np.float32)
    return result_flux, result_error


# The original PSF is also compressed by angular_scale.
# Add enough convolution to reach the requested target angular PSF.
compressed_fwhm_arcsec = input_fwhm_arcsec * angular_scale

kernel_fwhm_arcsec = np.sqrt(
    max(target_fwhm_arcsec**2 - compressed_fwhm_arcsec**2, 0.0)
)

kernel_sigma_pixels = (
    kernel_fwhm_arcsec
    / pixel_scale_arcsec
    / 2.354820045
)

scaled_channels = []
scaled_error_channels = []
original_sums = []
scaled_sums = []

with fits.open(input_fits, memmap=True) as hdul:
    for extension, error_extension in zip(line_extensions, error_extensions):
        original = np.asarray(
            hdul[extension].data,
            dtype=np.float32,
        )
        original_error = np.asarray(
            hdul[error_extension].data,
            dtype=np.float32,
        )

        if original.ndim != 2:
            raise ValueError(
                f"{extension} has shape {original.shape}; expected a 2-D map."
            )

        if original_error.shape != original.shape:
            raise ValueError(
                f"{error_extension} has shape {original_error.shape}; expected {original.shape}."
            )

        scaled, scaled_error = rescale_flux_and_error_about_center(
            original, original_error, scale=angular_scale
        )
        scaled, scaled_error = convolve_flux_and_error_preserving_nans(
            scaled, scaled_error,
            sigma_pixels=kernel_sigma_pixels,
        )

        scaled_channels.append(scaled)
        scaled_error_channels.append(scaled_error)
        original_sums.append(np.nansum(original))
        scaled_sums.append(np.nansum(scaled))


# Shape: (height, width, 4)
data = np.stack(scaled_channels, axis=-1).astype(np.float32)
error_data = np.stack(scaled_error_channels, axis=-1).astype(np.float32)

header = fits.Header()
header["GALAXY"] = "M33"
header["SIMDIST"] = (distance_ngc253, "Simulated distance [Mpc]")
header["ORIGDIST"] = (distance_m33, "Original distance [Mpc]")
header["ANGSCALE"] = (angular_scale, "Angular linear-size scale")
header["FLXSCALE"] = (expected_flux_scale, "Expected integrated flux scale")
header["NCHAN"] = (len(line_extensions), "Number of channels")
header["PIXSCALE"] = (pixel_scale_arcsec, "Output pixel scale [arcsec/pix]")
header["INPFWHM"] = (input_fwhm_arcsec, "Input PSF FWHM [arcsec]")
header["TARGFWHM"] = (target_fwhm_arcsec, "Target PSF FWHM [arcsec]")
header["KERNSIG"] = (kernel_sigma_pixels, "Added Gaussian sigma [pixel]")
header["ERRPROP"] = ("VARIANCE", "Errors propagated as independent variances")
header["CHAN1"] = "HB4861_FLUX"
header["CHAN2"] = "OIII5006_FLUX"
header["CHAN3"] = "HA6562_FLUX"
header["CHAN4"] = "NII6583_FLUX"

primary_hdu = fits.PrimaryHDU(data=data, header=header)
error_hdu = fits.ImageHDU(data=error_data, name="ERROR")
error_hdu.header["BUNIT"] = ("same as flux", "Per-pixel 1-sigma uncertainty")
fits.HDUList([primary_hdu, error_hdu]).writeto(output_fits, overwrite=True)

print("Output:", output_fits)
print("Shape:", data.shape)
print("Error shape:", error_data.shape)
print(f"Angular scale: {angular_scale:.5f}")
print(f"Expected integrated-flux ratio: {expected_flux_scale:.5f}")
print(f"Additional PSF sigma: {kernel_sigma_pixels:.3f} pixels")

for extension, old_sum, new_sum in zip(
    line_extensions,
    original_sums,
    scaled_sums,
):
    measured_ratio = new_sum / old_sum if old_sum != 0 else np.nan

    print(
        f"{extension:16s}: "
        f"original={old_sum:.6e}, "
        f"scaled={new_sum:.6e}, "
        f"ratio={measured_ratio:.5f}"
    )

Output: M33_4channel_NGC253_distance.fits
Shape: (2636, 894, 4)
Error shape: (2636, 894, 4)
Angular scale: 0.24000
Expected integrated-flux ratio: 0.05760
Additional PSF sigma: 1.649 pixels
HB4861_FLUX     : original=6.326975e+08, scaled=3.898094e+07, ratio=0.06161
OIII5006_FLUX   : original=5.409786e+08, scaled=3.333031e+07, ratio=0.06161
HA6562_FLUX     : original=2.035618e+09, scaled=1.256023e+08, ratio=0.06170
NII6583_FLUX    : original=5.257940e+08, scaled=3.574549e+07, ratio=0.06798
